# PortfolioVec demo

A step-by-step walk through the pipeline. Every result comes from vector addition and the inner product (Boyd & Vandenberghe, Chapter 1).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')          # run from the project root
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from src.portfoliovec import loader, validator, vectors, returns, compare

## Step 0: the worked example
`p = (10, 20)`, `s = (5, 3)`, `b = (2, -1)`

In [ ]:
p = np.array([10., 20.]); s = np.array([5., 3.]); b = np.array([2., -1.])
print('V     =', vectors.portfolio_value(p, s))
print('s_new =', vectors.apply_trade(s, b))
print('cash  =', vectors.trade_cash(p, b))
print('V_new =', vectors.portfolio_value(p, vectors.apply_trade(s, b)))

## Step 1-2: load and validate real prices
`P` is a T x n matrix: rows = days, columns = assets.

In [ ]:
cfg = yaml.safe_load(open('config.yaml'))
P, dates, assets = loader.load_prices(cfg)
validator.check(P, cfg)
print(P.shape)
pd.DataFrame(P, index=dates, columns=assets).tail()

## Step 3-4: vectors, value and trade
`V = pᵀs`, `s_new = s + b`, cash = `pᵀb`. Note the negative TSLA entry: a short position.

In [ ]:
s = np.array(cfg['holdings'], dtype=float)
b = np.array(cfg['trade'], dtype=float)
p = P[-1]
s_new = vectors.apply_trade(s, b)
v_before, v_after = vectors.portfolio_value(p, s), vectors.portfolio_value(p, s_new)
cash = vectors.trade_cash(p, b)
print(f'before {v_before:,.2f}   after {v_after:,.2f}   cash {cash:,.2f}')
print('check: after - before == cash ->', np.isclose(v_after - v_before, cash))
pd.DataFrame({'before': s, 'trade': b, 'after': s_new, 'price': p,
              'value before': p*s, 'value after': p*s_new}, index=assets).round(2)

## Step 5: weights and the return matrix
`w = h / 1ᵀh` sums to 1; the short gives a negative weight. `R = P[1:] / P[:-1] - 1`.

In [ ]:
w_A = vectors.weights_from_holdings(p, s)
w_B = np.array(cfg['allocations']['B']['weights'], dtype=float)
R = returns.return_matrix(P)
print('R shape:', R.shape, '| sum w_A =', w_A.sum().round(10), '| leverage A =', round(vectors.leverage(w_A), 3))
pd.DataFrame({'w_A': w_A, 'w_B': w_B}, index=assets).round(4)

## Step 6-7: portfolio returns and comparison
`ρ = R @ w` (one inner product per day), then `V_{t+1} = V_t (1 + ρ_t)`.

In [ ]:
allocs = {'A': {'name': cfg['allocations']['A']['name'], 'weights': w_A},
          'B': {'name': cfg['allocations']['B']['name'], 'weights': w_B}}
results = compare.compare_allocations(R, allocs, cfg['initial_capital'])
pd.DataFrame({k: r['summary'] for k, r in results.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for k, r in results.items():
    ax.plot(dates, r['path'], lw=2, label=f"{k}: {r['name']}")
ax.set_title('Portfolio value over time'); ax.set_ylabel('Value ($)'); ax.legend(); plt.show()

## Did the short help?
The short's daily contribution to A is `r_TSLA * w_TSLA`. It is positive on days TSLA falls.

In [ ]:
i = assets.index('TSLA')
contrib = R[:, i] * w_A[i]
print(f'TSLA fell on {(R[:, i] < 0).sum()} of {len(R)} days')
print(f'Total contribution of the short to A: {contrib.sum():+.2%}')
print('Winner:', compare.winner(results))